## Generate food-name token embeddings using 6 pre-trained language models

## 1. Setup

In [8]:
import os, json, re
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

# Load configuration
os.makedirs('embeddings', exist_ok=True)

with open('data/config.json', encoding='utf-8') as f:
    CONFIG = json.load(f)

# Select device
def select_device():
    if torch.cuda.is_available():
        return 'cuda'
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

DEVICE = select_device()

In [9]:
# Pretrained language models
ALL_MODELS = {
    'BERT':        'bert-base-multilingual-cased',
    'LegalBERT':   'nlpaueb/legal-bert-base-uncased',
    'DistilBERT':  'distilbert-base-multilingual-cased',
    'XLM-RoBERTa': 'xlm-roberta-base',
    'GPT-2':       'gpt2',
    'KLUE-BERT':   'klue/bert-base',
}

MAX_LENGTH = 32
BATCH_SIZE = 128

## 2. Embedding Functions

In [10]:
def mean_pooling(last_hidden_state, attention_mask):
    # Mean pooling
    mask = attention_mask.unsqueeze(-1).float()
    return (last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


@torch.no_grad()
def embed_tokens(tokens, model_name):
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()

    # Set padding token if needed
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Generate embeddings by batch
    batches = []
    for start in range(0, len(tokens), BATCH_SIZE):
        batch = [str(token) for token in tokens[start:start + BATCH_SIZE]]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt'
        ).to(DEVICE)

        hidden = model(**encoded).last_hidden_state
        batches.append(
            mean_pooling(hidden, encoded['attention_mask']).cpu().numpy()
        )

    # Clear model from memory
    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    elif DEVICE == 'mps':
        torch.mps.empty_cache()

    return np.vstack(batches).astype(np.float32)


def safe_name(value):
    # Convert to a file-safe name
    return re.sub(r'[^0-9A-Za-z가-힣]+', '_', str(value)).strip('_')

## 3. Prepare Tokens

In [11]:
KO = CONFIG['KO']
TOKEN_COL = CONFIG['TOKEN_COL']

# Load train and test data
files = [
    'data/KNHANES_2024_train.csv',
    'data/KNHANES_2024_test.csv',
]

frames = [pd.read_csv(path, low_memory=False) for path in files]
merged = pd.concat(frames, ignore_index=True)

# Collect unique food-name tokens
unique_tokens = sorted({
    token
    for text in merged[TOKEN_COL].dropna()
    for token in str(text).split(',')
    if token
})

print(f'Unique tokens: {len(unique_tokens):,}')

Unique tokens: 2,368


## 4. Generate Embeddings

In [12]:
import time

# Save token vocabulary
token_path = f'embeddings/{safe_name(KO)}__tokens.csv'
pd.DataFrame({'token': unique_tokens}).to_csv(
    token_path, index=False, encoding='utf-8-sig'
)
print(f'Saved: {token_path}')

# Generate embeddings for each model
for model_label, model_name in ALL_MODELS.items():
    print(f'\n{model_label}')

    vector_path = f'embeddings/{safe_name(KO)}__{safe_name(model_label)}.npy'

    if os.path.exists(vector_path):
        print('Skipped: file already exists')
        continue

    start = time.time()
    vectors = embed_tokens(unique_tokens, model_name)
    np.save(vector_path, vectors)

    elapsed = time.time() - start
    print(f'Saved: {vector_path} | shape={vectors.shape} | {elapsed:.1f}s')

Saved: embeddings/식품코드명_한글__tokens.csv

BERT


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved: embeddings/식품코드명_한글__BERT.npy | shape=(2368, 768) | 7.0s

LegalBERT


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Saved: embeddings/식품코드명_한글__GPT_2.npy | shape=(2368, 768) | 9.8s

KLUE-BERT


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved: embeddings/식품코드명_한글__KLUE_BERT.npy | shape=(2368, 768) | 4.7s
